# Phase 1 Task B — Machine B: Swin-Tiny + ResNet-50 (PARALLEL)

**Assignment:**
- Swin-Tiny × seeds [42, 123, 456]
- ResNet-50 × seeds [42, 123, 456]

**2 models chạy SONG SONG trên 1× RTX 4090 24GB**

Swin batch=16 (nặng), ResNet batch=16. Tổng VRAM ~14 GB — thoải mái.

## 1. Setup & Root Detection <a id='1-setup'></a>

In [ ]:
from pathlib import Path

NOTEBOOK = Path("__file__").resolve()
ROOT = NOTEBOOK
for _ in range(10):
    ROOT = ROOT.parent
    if (ROOT / "pyproject.toml").exists() or (ROOT / ".git").exists():
        break
print(f"[ROOT] {ROOT}")
import sys, os
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print(f"[CWD] {os.getcwd()}")

## 2. Dependencies & wandb Auth

In [ ]:
import subprocess, sys, os, getpass

PKGS = ["timm", "albumentations", "scikit-learn", "seaborn", "wandb", "ptflops"]
for pkg in PKGS:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        print(f"[Install] {pkg}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

if "WANDB_API_KEY" not in os.environ:
    os.environ["WANDB_API_KEY"] = getpass.getpass("wandb API key: ")
print("[wandb] Ready")

## 3. Prepare Dataset

In [ ]:
HF_REPO = "ny0g507h3p/wood-task-B"
DATA_ROOT = ROOT / "data" / "task-B"

if not (DATA_ROOT / "train").exists():
    print("[HF] Downloading dataset...")
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=HF_REPO, repo_type="dataset",
                     local_dir=str(DATA_ROOT), ignore_patterns=["*.md", "README*"])
else:
    print(f"[Data] {DATA_ROOT} exists")

from src.data import get_dataloaders
tl, vl, testl = get_dataloaders(str(DATA_ROOT), batch_size=16, num_workers=4, task="taskB")
print(f"[Data] Train={len(tl.dataset)}, Val={len(vl.dataset)}, Test={len(testl.dataset)}")
print(f"[Data] Classes={len(tl.dataset.classes)}")

## 4. Training Helpers — Parallel Runner

In [ ]:
import wandb, torch, threading, queue
from src.models import create_model
from src.train import TrainingEngine, create_optimizer, create_scheduler, create_loss_function
from src.utils import set_seed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Device] {DEVICE}")
if torch.cuda.is_available():
    print(f"[GPU] {torch.cuda.get_device_name(0)}")
    print(f"[VRAM] {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
def train_one_in_thread(
    backbone: str, seed: int, data_root: str,
    batch_size: int, epochs: int,
    result_queue: queue.Queue, lock: threading.Lock,
    label: str,
):
    """Train on the shared GPU; batch_size=16 keeps 2 models ~14 GB total."""
    set_seed(seed)
    output_dir = ROOT / "experiments" / "taskB" / backbone / f"seed_{seed}"
    output_dir.mkdir(parents=True, exist_ok=True)

    train_loader, val_loader, _ = get_dataloaders(
        data_root=data_root, batch_size=batch_size, num_workers=4, task="taskB",
    )
    num_classes = len(train_loader.dataset.classes)

    with lock:
        print(f"[{label}] {backbone} s{seed} | {len(train_loader.dataset)} train | batch={batch_size}")

    wandb_run = wandb.init(
        project="woodbench-taskB",
        name=f"{backbone}_s{seed}_machine-B",
        config={
            "backbone": backbone, "seed": seed,
            "epochs": epochs, "batch_size": batch_size,
            "num_classes": num_classes,
            "lr_head": 1e-4, "lr_backbone": 1e-5,
            "label_smoothing": 0.1,
            "scheduler": "cosine", "amp": True,
            "machine": "B", "parallel": True,
        },
        dir=str(output_dir),
    )

    model = create_model(backbone, num_classes=num_classes,
                        pretrained=True, freeze_backbone=False)
    optimizer = create_optimizer(model, lr=1e-4, weight_decay=1e-4, backbone_lr=1e-5)
    scheduler = create_scheduler(optimizer, epochs=epochs, scheduler_type="cosine")
    criterion = create_loss_function("label_smoothing", label_smoothing=0.1)

    engine = TrainingEngine(
        model=model, train_loader=train_loader, val_loader=val_loader,
        optimizer=optimizer, scheduler=scheduler, criterion=criterion,
        device=DEVICE, use_amp=True,
    )

    result = engine.fit(
        epochs=epochs, output_dir=output_dir,
        name=f"{backbone}_seed{seed}", save_best_only=False,
    )

    wandb_run.finish()
    with lock:
        print(f"[{label}] {backbone} s{seed} DONE — val_acc={result['best_val_acc']:.2f}%")
    result_queue.put({"backbone": backbone, "seed": seed, "result": result})

In [ ]:
def run_parallel_pair(
    backbone_a, seed_a, backbone_b, seed_b,
    data_root, batch_a=16, batch_b=16, epochs=50,
):
    print(f"\n{'='*65}")
    print(f"PARALLEL: {backbone_a} s{seed_a}  ||  {backbone_b} s{seed_b}")
    print(f"         batch {batch_a}           batch {batch_b}")
    print(f"{'='*65}")

    result_queue = queue.Queue()
    lock = threading.Lock()
    threads = []

    t_a = threading.Thread(
        target=train_one_in_thread,
        args=(backbone_a, seed_a, data_root, batch_a, epochs,
              result_queue, lock, "GPU-A"),
        name=f"{backbone_a}_s{seed_a}",
    )
    t_b = threading.Thread(
        target=train_one_in_thread,
        args=(backbone_b, seed_b, data_root, batch_b, epochs,
              result_queue, lock, "GPU-B"),
        name=f"{backbone_b}_s{seed_b}",
    )

    t_a.start()
    t_b.start()
    t_a.join()
    t_b.join()

    results = [result_queue.get(), result_queue.get()]
    print(f"\n[PAIR DONE] {backbone_a} s{seed_a} + {backbone_b} s{seed_b}")
    return results

## 5. Run All Seeds — Swin || ResNet

In [ ]:
SEEDS = [42, 123, 456]
all_results = []

for seed in SEEDS:
    pair_results = run_parallel_pair(
        backbone_a="swin_tiny",    seed_a=seed,
        backbone_b="resnet50",     seed_b=seed,
        data_root=str(DATA_ROOT),
        batch_a=16,
        batch_b=16,
        epochs=50,
    )
    all_results.extend(pair_results)

print("\n" + "="*65)
print("MACHINE B — PARALLEL RUN COMPLETE")
print("="*65)
for r in all_results:
    print(f"  {r['backbone']} s{r['seed']}: {r['result']['best_val_acc']:.2f}%")

## 6. Summary

In [ ]:
import numpy as np
from collections import defaultdict

by_model = defaultdict(list)
for r in all_results:
    by_model[r["backbone"]].append(r["result"]["best_val_acc"])

print("="*60)
print("MACHINE B — FINAL SUMMARY")
print("="*60)
for model, accs in by_model.items():
    print(f"\n{model}:")
    for seed, acc in zip(SEEDS, accs):
        print(f"  seed={seed}: {acc:.2f}%")
    print(f"  Mean: {np.mean(accs):.2f}% ± {np.std(accs):.2f}%")

print(f"\n[wandb] All runs → woodbench-taskB (tag machine=B)")
print(f"[Next] Machine A: EfficientNet sau khi ConvNeXt+MobileNet xong")